# Adaptive Visual Token Pruning Engine for Low-Token Document OCR

This notebook implements an end-to-end framework for dynamic visual token pruning and bipartite token merging (**ToMe**) for Document OCR (**Donut**).

### Kaggle Setup Instructions:
1. **GPU Accelerator:** Select `GPU T4 x2` or `GPU P100` under Notebook Options on the right sidebar.
2. **Internet:** Toggle `Internet` to **On** on the right sidebar.
3. Click **Run All** to train and benchmark the model.

In [ ]:
# Cell 1: Environment Installation
!pip install -q "transformers>=4.38.0" "datasets>=2.16.0" "accelerate>=0.27.0" sentencepiece editdistance matplotlib pillow torchvision

In [ ]:
# Cell 2: Imports, GPU Verification & Memory Optimization
import os
import gc
import time
import json
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from tqdm.auto import tqdm
from datasets import load_dataset
from transformers import VisionEncoderDecoderModel, DonutProcessor
from transformers.modeling_outputs import BaseModelOutput

# Enable expandable memory segments to avoid CUDA memory fragmentation
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

# Clean stale memory allocations
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

# A CPU session is always a mistake here, and it does not announce itself: the
# 2026-08-29 attempt printed "Using device: cpu" and kept going. Training would
# take weeks; even eval-only decoding is hours instead of ~11 min, and CPU/GPU
# numerics differ enough to trip the ablation's 0.5pt CONTROL check against
# run 5's GPU-produced reference. Fail immediately instead.
ALLOW_CPU = False  # set True only for a deliberate no-GPU smoke test
assert ALLOW_CPU or device.type == 'cuda', (
    'No GPU: Kaggle sidebar -> Session options -> Accelerator -> GPU T4 x2, '
    'then Run All. (Set ALLOW_CPU=True to override, expect hours not minutes.)')

if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    free_mem, total_mem = torch.cuda.mem_get_info()
    print(f'Available VRAM: {free_mem / (1024**3):.2f} GB / {total_mem / (1024**3):.2f} GB')

# ---- Phase 2c: eval-only resume (set this to skip the ~4h retrain) ----
# Point RESUME_CKPT at run 5's uploaded checkpoint to go straight to eval + the
# decoding ablation; leave it None to train normally. Validated HERE, before the
# ~19-minute dataset download in Cell 6, so a wrong path costs a minute.
RESUME_CKPT = None  # e.g. '/kaggle/input/adaptive-donut-run5/adaptive_donut_funsd.pt'

if RESUME_CKPT and not os.path.exists(RESUME_CKPT):
    # The path is wrong in one of three ordinary ways: the dataset was uploaded
    # with its checkpoints/ folder (file sits one level deeper), the slug differs
    # from the dataset title, or the dataset was never attached. Resolve by
    # searching for the basename instead of guessing the shape.
    _want = os.path.basename(RESUME_CKPT)
    _found = [os.path.join(r, f)
              for r, _d, fs in os.walk('/kaggle/input') for f in fs if f == _want]
    if len(_found) == 1:
        print(f'RESUME_CKPT not at {RESUME_CKPT}\n  -> resolved to {_found[0]}')
        RESUME_CKPT = _found[0]
    else:
        _all_pt = [os.path.join(r, f)
                   for r, _d, fs in os.walk('/kaggle/input') for f in fs if f.endswith('.pt')]
        raise AssertionError(
            f'RESUME_CKPT not found: {RESUME_CKPT}\n'
            f'  matches for {_want!r}: {_found or "none"}\n'
            f'  .pt files mounted under /kaggle/input: {_all_pt or "NONE -- dataset not attached?"}\n'
            f'  top level of /kaggle/input: {sorted(os.listdir("/kaggle/input")) if os.path.isdir("/kaggle/input") else "MISSING"}\n'
            '  Fix: Add Input -> your dataset, then copy the path it shows.')

EVAL_ONLY = bool(RESUME_CKPT)
if EVAL_ONLY:
    print(f'EVAL-ONLY mode: will load {RESUME_CKPT}')
    print('  -> training skipped, and Cell 6 will skip building the train set')

## 1. Core Architecture Modules
- **`PatchSaliencyRouter`:** Predicts patch importance scores and applies a Straight-Through Estimator (STE) top-k filter.
- **`BipartiteTokenMerger`:** Fully vectorized, non-in-place soft token merging via matrix operations.

In [ ]:
# Cell 3: Patch Saliency Router and Vectorized ToMe Modules
class PatchSaliencyRouter(nn.Module):
    def __init__(self, hidden_dim: int = 1024, reduction_dim: int = 256):
        super().__init__()
        self.scorer = nn.Sequential(
            nn.Linear(hidden_dim, reduction_dim),
            nn.LayerNorm(reduction_dim),
            nn.GELU(),
            nn.Linear(reduction_dim, 1),
            nn.Sigmoid()
        )
        nn.init.constant_(self.scorer[-2].bias, 0.5)

    def forward(self, tokens, keep_ratio=0.35, coords=None, use_ste=True,
                select_scores=None, invert=False):
        """select_scores: (B, N) ranking signal used INSTEAD of the learned scores
        (uniform noise for a random control, patch ink for an oracle, a stratified
        key for a per-row budget). The learned `scores` are still computed and
        returned, so reporting is unaffected. invert: take the LOWEST-ranked
        tokens. Both are eval-only; see AGENTS.md Diagnostic D1/D2."""
        B, N, D = tokens.shape
        K = max(1, int(round(N * keep_ratio)))
        scores = self.scorer(tokens)  # (B, N, 1)
        
        if select_scores is not None:
            if select_scores.shape[:2] != (B, N):
                raise ValueError(
                    f'select_scores must be (B, N) = {(B, N)}, got {tuple(select_scores.shape)}')
            rank_by = select_scores.reshape(B, N).to(scores.dtype)
        else:
            rank_by = scores.squeeze(-1)
        topk_scores, topk_indices = torch.topk(rank_by, k=K, dim=1, largest=not invert, sorted=True)
        indices_expanded = topk_indices.unsqueeze(-1).expand(-1, -1, D)
        selected_tokens = torch.gather(tokens, dim=1, index=indices_expanded)
        
        # STE is skipped under an external ranking: topk_scores would then hold the
        # override's values, not the scorer's, so scaling tokens by them sends the
        # router head a gradient it is not responsible for.
        if self.training and use_ste and select_scores is None:
            ste_weights = topk_scores.unsqueeze(-1)
            ste_multiplier = 1.0 + (ste_weights - ste_weights.detach())
            selected_tokens = selected_tokens * ste_multiplier
            
        pruned_coords = None
        if coords is not None:
            coords_expanded = topk_indices.unsqueeze(-1).expand(-1, -1, coords.shape[-1])
            pruned_coords = torch.gather(coords, dim=1, index=coords_expanded)
            
        return selected_tokens, scores, topk_indices, pruned_coords


class BipartiteTokenMerger(nn.Module):
    def __init__(self, hidden_dim: int = 1024):
        super().__init__()
        self.hidden_dim = hidden_dim

    def forward(self, tokens, merge_ratio=0.20, coords=None):
        B, K, D = tokens.shape
        if K <= 4 or merge_ratio <= 0.0:
            return tokens, coords
            
        r = min(int(round(K * merge_ratio)), K // 2)
        if r <= 0:
            return tokens, coords
            
        idx_A = torch.arange(0, K, 2, device=tokens.device)
        idx_B = torch.arange(1, K, 2, device=tokens.device)
        
        A = tokens[:, idx_A, :]
        B_set = tokens[:, idx_B, :]
        coords_A = coords[:, idx_A, :] if coords is not None else None
        coords_B = coords[:, idx_B, :] if coords is not None else None
        
        K_A, K_B = A.shape[1], B_set.shape[1]
        r_actual = min(r, K_A)
        
        A_norm = F.normalize(A, p=2, dim=-1)
        B_norm = F.normalize(B_set, p=2, dim=-1)
        sim = torch.bmm(A_norm, B_norm.transpose(1, 2))
        
        max_sim_val, max_sim_idx = sim.max(dim=-1)
        _, top_r_A_idx = torch.topk(max_sim_val, k=r_actual, dim=-1, largest=True)
        
        T = torch.zeros((B, K_A, K_B), device=tokens.device, dtype=tokens.dtype)
        batch_idx = torch.arange(B, device=tokens.device).unsqueeze(1).expand(B, r_actual)
        target_b = torch.gather(max_sim_idx, 1, top_r_A_idx)
        T[batch_idx, top_r_A_idx, target_b] = 1.0
        
        T_t = T.transpose(1, 2)
        W_B = 1.0 + T_t.sum(dim=-1, keepdim=True)
        B_merged = (B_set + torch.bmm(T_t, A)) / W_B
        
        coords_B_merged = None
        if coords is not None:
            coords_B_merged = (coords_B + torch.bmm(T_t, coords_A)) / W_B
            
        mask_A = torch.ones((B, K_A), dtype=torch.bool, device=tokens.device)
        mask_A[batch_idx, top_r_A_idx] = False
        
        A_unmerged = torch.stack([A[b, mask_A[b]] for b in range(B)], dim=0)
        coords_A_unmerged = torch.stack([coords_A[b, mask_A[b]] for b in range(B)], dim=0) if coords is not None else None
        
        out_tokens = torch.cat([B_merged, A_unmerged], dim=1)
        out_coords = torch.cat([coords_B_merged, coords_A_unmerged], dim=1) if coords is not None else None
        return out_tokens, out_coords

In [ ]:
# Cell 4: Custom Multi-Task Sparsity Loss
class AdaptivePruningLoss(nn.Module):
    def __init__(self, lambda_sparsity=2.0, lambda_entropy=0.05, target_budget=0.25, pad_token_id=-100):
        super().__init__()
        self.lambda_sparsity = lambda_sparsity
        self.lambda_entropy = lambda_entropy
        self.target_budget = target_budget
        self.ce_loss_fn = nn.CrossEntropyLoss(ignore_index=pad_token_id)

    def forward(self, decoder_logits, labels, scores, provided_ce_loss=None):
        if provided_ce_loss is not None:
            ce_loss = provided_ce_loss
        elif decoder_logits is not None and labels is not None:
            vocab_size = decoder_logits.shape[-1]
            ce_loss = self.ce_loss_fn(decoder_logits.view(-1, vocab_size), labels.view(-1))
        else:
            ce_loss = torch.tensor(0.0, device=scores.device)

        mean_score = scores.mean()
        target_tensor = torch.tensor(self.target_budget, device=scores.device, dtype=scores.dtype)
        sparsity_loss = F.smooth_l1_loss(mean_score, target_tensor)

        eps = 1e-7
        clamped_scores = scores.clamp(min=eps, max=1.0 - eps)
        entropy = -(clamped_scores * torch.log(clamped_scores) + (1.0 - clamped_scores) * torch.log(1.0 - clamped_scores)).mean()
        entropy_loss = -entropy

        total_loss = ce_loss + self.lambda_sparsity * sparsity_loss + self.lambda_entropy * entropy_loss
        return {
            'loss': total_loss,
            'ce_loss': ce_loss,
            'sparsity_loss': sparsity_loss,
            'mean_score': mean_score.detach()
        }

## 2. Integrated Model Architecture (`AdaptiveDonutOCR`)
Freezes the Swin vision encoder to eliminate 14 GB of attention graph storage, training the router and text decoder with under 2 GB VRAM.

In [ ]:
# Cell 5: AdaptiveDonutOCR Model
SELECT_MODES = ('router', 'negated', 'random', 'ink', 'stratified', 'stratified_negated')

TOKEN_GRID = (80, 60)   # Swin-B at 2560x1920, total stride 32 -> 80*60 = 4800 tokens


def patch_ink(pixel_values, num_tokens, stride=32):
    """Per-token within-patch contrast on the encoder's own token grid -> (B, N).

    A learning-free proxy for 'there is text here', used as the ink oracle's ranking
    signal and as the retained-ink diagnostic.

    Within-patch std rather than mean darkness because std is blind to whether Donut
    padded with black or white -- uniform padding has zero contrast either way, while
    darkness would score black padding as maximum ink. No un-normalization is needed:
    donut-base uses image_mean = image_std = 0.5 on every channel, so pixel_values is
    a uniform affine image of luminance and a plain channel mean ranks identically to
    true grayscale.
    """
    B, _, H, W = pixel_values.shape
    gh, gw = H // stride, W // stride
    if gh * gw != num_tokens:
        raise ValueError(
            f'token grid mismatch: pixel_values {H}x{W} at stride {stride} gives '
            f'{gh}x{gw}={gh * gw} patches but the encoder emitted {num_tokens} tokens')
    gray = pixel_values.mean(dim=1)                                    # (B, H, W)
    blocks = gray.unfold(1, stride, stride).unfold(2, stride, stride)   # (B,gh,gw,s,s)
    return blocks.reshape(B, gh * gw, -1).std(dim=-1)                  # (B, N)


def stratified_scores(scores, grid=TOKEN_GRID, negate=False):
    """Rewrite a ranking so a GLOBAL top-K becomes a PER-GRID-ROW top-k.

    Diagnostic D2: a single global top-k has no mechanism bounding worst-case
    coverage. It may spend the entire budget on one dense region, and the trained
    router does something like this -- its MINIMUM text-line ink coverage is 0.000,
    i.e. at least one line of the page loses all its ink, while total retained ink
    looks unremarkable. Recall is gated by the worst line, not the average.

    A per-row budget makes starving a line structurally impossible. Rather than add a
    second selection path to the router (a new path is a new place for the two to
    diverge), stratification is expressed purely as a monotone rewrite of the ranking
    signal, reusing the select_scores machinery unchanged:

        key = (gw - within_row_rank) + normalized_score

    within_row_rank is an integer in [0, gw), so the integer part dominates and the
    normalized score only breaks ties INSIDE a rank band; the bands cannot overlap
    because the score term is clamped strictly below 1.0. A global top-K with
    K = gh * k therefore selects exactly the top-k of every row. When K is not a
    multiple of gh it degrades gracefully -- whole bands fill first, the remainder
    goes to the best scores in the next band, and rows differ by at most 1.

    negate ranks by the negated score WITHIN each row, which is the fix for a
    sign-inverted scorer. Stratification and sign are orthogonal -- hence two modes.
    """
    B, N = scores.shape
    gh, gw = grid
    if gh * gw != N:
        raise ValueError(
            f'token grid mismatch: grid {gh}x{gw}={gh * gw} but got {N} tokens')
    s = (-scores if negate else scores).reshape(B, gh, gw)
    order = s.argsort(dim=-1, descending=True)
    rank = torch.empty_like(order)
    rank.scatter_(-1, order, torch.arange(gw, device=s.device).expand_as(order))
    lo = s.amin(dim=(1, 2), keepdim=True)
    hi = s.amax(dim=(1, 2), keepdim=True)
    s_norm = ((s - lo) / (hi - lo).clamp(min=1e-9)).clamp(0.0, 0.999)
    return ((gw - rank).to(s.dtype) + s_norm).reshape(B, N)


class AdaptiveDonutOCR(nn.Module):
    def __init__(self, base_model_name='naver-clova-ix/donut-base', keep_ratio=0.35, merge_ratio=0.20, freeze_encoder=True):
        super().__init__()
        self.base_model_name = base_model_name
        self.keep_ratio = keep_ratio
        self.merge_ratio = merge_ratio
        
        # Clean cache
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            
        print(f'Loading base backbone {base_model_name}...')
        self.model = VisionEncoderDecoderModel.from_pretrained(
            base_model_name,
            low_cpu_mem_usage=True
        )
        
        # Freeze vision encoder to eliminate the 14 GB Swin attention memory graph
        if freeze_encoder:
            print('Vision backbone frozen (saves ~12 GB VRAM). Training Router & Decoder!')
            for param in self.model.encoder.parameters():
                param.requires_grad = False
                
        hidden_dim = self.model.config.encoder.hidden_size
        self.router = PatchSaliencyRouter(hidden_dim=hidden_dim)
        self.tome_merger = BipartiteTokenMerger(hidden_dim=hidden_dim)

    def _generate_2d_coords(self, B, N, device, dtype=torch.float32):
        aspect_ratio = 4.0 / 3.0
        W_grid = int(round((N * aspect_ratio) ** 0.5))
        H_grid = max(1, N // W_grid)
        grid_y, grid_x = torch.meshgrid(
            torch.linspace(0, 1, H_grid, device=device, dtype=dtype),
            torch.linspace(0, 1, W_grid, device=device, dtype=dtype),
            indexing='ij'
        )
        coords_2d = torch.stack([grid_x, grid_y], dim=-1).view(-1, 2)
        if coords_2d.shape[0] < N:
            pad = coords_2d[-1:].repeat(N - coords_2d.shape[0], 1)
            coords_2d = torch.cat([coords_2d, pad], dim=0)
        else:
            coords_2d = coords_2d[:N]
        return coords_2d.unsqueeze(0).expand(B, -1, -1)

    def forward(self, pixel_values, labels=None, decoder_input_ids=None, keep_ratio=None, merge_ratio=None):
        k_ratio = keep_ratio if keep_ratio is not None else self.keep_ratio
        m_ratio = merge_ratio if merge_ratio is not None else self.merge_ratio
        B = pixel_values.shape[0]
        device = pixel_values.device
        
        # Forward pass through encoder (under no_grad when frozen to save 12GB)
        encoder_is_frozen = not any(p.requires_grad for p in self.model.encoder.parameters())
        if encoder_is_frozen:
            with torch.no_grad():
                encoder_outputs = self.model.encoder(pixel_values)
        else:
            encoder_outputs = self.model.encoder(pixel_values)
            
        visual_tokens = encoder_outputs.last_hidden_state
        N = visual_tokens.shape[1]
        
        coords = self._generate_2d_coords(B, N, device, dtype=visual_tokens.dtype)
        selected_tokens, scores, topk_indices, pruned_coords = self.router(
            visual_tokens, keep_ratio=k_ratio, coords=coords, use_ste=self.training
        )
        
        compressed_tokens, _ = self.tome_merger(
            selected_tokens, merge_ratio=m_ratio, coords=pruned_coords
        )
        M = compressed_tokens.shape[1]
        
        # NOTE: self.model.decoder is an MBartForCausalLM, which does NOT shift
        # labels internally. Feeding decoder_input_ids == labels and letting the
        # decoder compute the loss trains the model to COPY the current token
        # (next-token loss == 0) and collapses generation to a repeated token.
        # Fix: get raw logits, then compute the loss with an explicit right-shift
        # so position t predicts token t+1 (standard autoregressive objective).
        decoder_outputs = self.model.decoder(
            input_ids=decoder_input_ids,
            encoder_hidden_states=compressed_tokens,
            return_dict=True
        )
        logits = decoder_outputs.logits
        
        loss = None
        if labels is not None:
            shift_logits = logits[:, :-1, :].contiguous()
            shift_labels = labels[:, 1:].contiguous()
            loss = F.cross_entropy(
                shift_logits.view(-1, shift_logits.size(-1)),
                shift_labels.view(-1),
                ignore_index=-100
            )
        
        return {
            'loss': loss,
            'logits': logits,
            'scores': scores,
            'topk_indices': topk_indices,
            'original_tokens': N,
            'compressed_tokens': M,
            'compression_ratio': (1.0 - (M / N)) * 100.0
        }

    @torch.no_grad()
    def generate(self, pixel_values, decoder_input_ids=None, max_length=256,
                 keep_ratio=None, merge_ratio=None, select_mode='router', **kwargs):
        k_ratio = keep_ratio if keep_ratio is not None else self.keep_ratio
        m_ratio = merge_ratio if merge_ratio is not None else self.merge_ratio
        B = pixel_values.shape[0]
        device = pixel_values.device
        
        encoder_outputs = self.model.encoder(pixel_values)
        visual_tokens = encoder_outputs.last_hidden_state
        N = visual_tokens.shape[1]
        
        # select_mode swaps the RANKING SIGNAL only. K, the architecture and the
        # decoder are untouched, so every row costs the same and stays comparable.
        #   router  -- the learned scores (the actual model)
        #   negated -- the same scores, lowest-first. D1 found the trained router
        #              anti-correlated with ink, so this retains MORE text (0.62-0.66
        #              vs 0.34-0.38) -- yet loses to random, so ink is necessary and
        #              not sufficient.
        #   random  -- uniform noise: the floor the router must beat. Drawn fresh per
        #              image from the ambient RNG (one fixed mask is a positional
        #              prior, a different and easier target); seed in the eval loop.
        #   ink     -- patch contrast, no learning: an ORACLE bounding what any
        #              ink-seeking router could reach on this decoder.
        #   stratified / stratified_negated -- the SAME learned scores under a
        #              per-grid-row budget instead of one global top-k (D2). Tests
        #              whether the selection RULE is the bug rather than the scorer.
        if select_mode not in SELECT_MODES:
            raise ValueError(f'select_mode must be one of {SELECT_MODES}, got {select_mode!r}')
        select_scores = None
        invert = False
        if select_mode == 'negated':
            invert = True
        elif select_mode == 'random':
            select_scores = torch.rand(B, N, device=device, dtype=visual_tokens.dtype)
        elif select_mode == 'ink':
            select_scores = patch_ink(pixel_values, N).to(visual_tokens.dtype)
        elif select_mode in ('stratified', 'stratified_negated'):
            # Grid derived from pixel_values, not assumed, so a non-standard input
            # raises here instead of silently stratifying over the wrong axis.
            grid = (pixel_values.shape[-2] // 32, pixel_values.shape[-1] // 32)
            raw = self.router.scorer(visual_tokens).squeeze(-1)
            select_scores = stratified_scores(
                raw, grid=grid, negate=(select_mode == 'stratified_negated')
            ).to(visual_tokens.dtype)

        coords = self._generate_2d_coords(B, N, device, dtype=visual_tokens.dtype)
        selected_tokens, scores, topk_indices, pruned_coords = self.router(
            visual_tokens, keep_ratio=k_ratio, coords=coords, use_ste=False,
            select_scores=select_scores, invert=invert
        )
        
        compressed_tokens, _ = self.tome_merger(
            selected_tokens, merge_ratio=m_ratio, coords=pruned_coords
        )
        M = compressed_tokens.shape[1]
        
        custom_encoder_output = BaseModelOutput(last_hidden_state=compressed_tokens)
        
        # Prime the decoder with the task prompt (<s_doc>) exactly as trained.
        # donut-base's config leaves decoder_start/eos/pad unset, so pass them
        # explicitly; otherwise generation cannot stop and runs to max_length.
        gen_kwargs = dict(kwargs)
        if decoder_input_ids is not None:
            gen_kwargs['decoder_input_ids'] = decoder_input_ids
        else:
            gen_kwargs.setdefault('decoder_start_token_id', self.model.config.decoder_start_token_id or 0)
        gen_kwargs.setdefault('eos_token_id', 2)
        gen_kwargs.setdefault('pad_token_id', 1)
        # Decoding defaults set by the Phase 2c ablation (50 FUNSD docs, run-5
        # weights, CONTROL row reproduced run 5 to 0.00 pt). See
        # results/ablation_decoding.json.
        #
        # repetition_penalty MUST stay 1.0. At the old 1.3 it capped output at
        # 71% of gold length and cost 27.2 pts of recall: HF penalizes every
        # token already emitted, but a full-page form legitimately re-uses its
        # whole vocabulary, so after ~80 words every plausible continuation is
        # downweighted while EOS -- never emitted -- is not. EOS wins and the
        # model stops mid-page. It also ate the repeated '"' in the JSON prefix
        # (valid JSON 0% -> 82% once removed).
        #
        # no_repeat_ngram_size stays 3. It was a no-op at rp=1.3 (nothing
        # repeated for it to block), but with rp=1.0 it is the guard against
        # greedy repetition collapse: dropping it over-generates to 124.6% of
        # gold, sends 22% of docs into max_length, and costs 6.7 pts of recall.
        gen_kwargs.setdefault('repetition_penalty', 1.0)
        gen_kwargs.setdefault('no_repeat_ngram_size', 3)
        
        generated_ids = self.model.generate(
            encoder_outputs=custom_encoder_output,
            max_length=max_length,
            **gen_kwargs
        )
        
        # Retained ink: what fraction of the page's contrast survived pruning.
        # Logged per call because it is the number that EXPLAINS an accuracy row --
        # without it a bad row reads as 'pruning hurts' rather than 'we pruned the
        # text'. Guarded: a non-standard grid must not invalidate the accuracy row.
        retained_ink = None
        try:
            ink = patch_ink(pixel_values, N)
            kept = ink.gather(1, topk_indices).sum(dim=1)
            retained_ink = float((kept / ink.sum(dim=1).clamp(min=1e-9)).mean())
        except ValueError:
            pass

        return generated_ids, {
            'original_tokens': N,
            'compressed_tokens': M,
            'compression_ratio': (1.0 - (M / N)) * 100.0,
            'select_mode': select_mode,
            'retained_ink': retained_ink,
            'scores': scores,
            'topk_indices': topk_indices
        }

## 3. Dataset Loading & Preprocessing
Loads English document dataset (`nielsr/funsd`) and formats samples for Donut.

In [ ]:
# Cell 6: Data Pipeline
from torchvision import transforms as T

MAX_WORDS = 128

# Phase 1.5: document-appropriate augmentation (TRAIN split only) to fight the
# 149-image overfit. Mild geometric + photometric jitter only -- NO flips or
# aggressive crops, which would destroy or drop text. fill=255 keeps the paper
# background white where rotation/scale expose the corners.
DOC_AUG = T.Compose([
    T.RandomApply([T.RandomRotation(degrees=2, fill=255)], p=0.5),
    T.ColorJitter(brightness=0.2, contrast=0.2),
    T.RandomApply([T.RandomAffine(degrees=0, scale=(0.9, 1.05), fill=255)], p=0.3),
    T.RandomApply([T.GaussianBlur(kernel_size=3, sigma=(0.1, 1.5))], p=0.3),
])


def reading_order_words(words, bboxes):
    """Sort words top-to-bottom, left-to-right using boxes [x0,y0,x1,y1].
    FUNSD stores words in form-annotation order, not reading order, so a naive
    join yields a scrambled target the decoder cannot learn or be scored on."""
    if not bboxes or len(bboxes) != len(words):
        return list(words)
    hs = sorted(b[3] - b[1] for b in bboxes if b[3] > b[1])
    tol = max(1.0, (hs[len(hs) // 2] * 0.6) if hs else 10.0)
    lines = []
    for i in sorted(range(len(words)), key=lambda k: bboxes[k][1]):
        top = bboxes[i][1]
        for anchor, idxs in lines:
            if abs(top - anchor) <= tol:
                idxs.append(i)
                break
        else:
            lines.append((top, [i]))
    out = []
    for _, idxs in lines:
        out.extend(sorted(idxs, key=lambda k: bboxes[k][0]))
    return [words[i] for i in out]


from torch.utils.data import ConcatDataset


def build_target_text(sample):
    '''Unified target across mixed sources -> the words for {"text": ...}, capped
    at MAX_WORDS. FUNSD: reading-ordered words. SynthDoG-en: gt_parse.text_sequence.
    Keeps the decoder on ONE output schema.'''
    words = sample.get('words')
    if words:
        boxes = sample.get('bboxes') or sample.get('boxes')
        if boxes:
            words = reading_order_words(words, boxes)
        return ' '.join(words[:MAX_WORDS])
    gt = sample.get('ground_truth')
    if gt:
        try:
            obj = json.loads(gt) if isinstance(gt, str) else gt
            parse = obj.get('gt_parse', obj) if isinstance(obj, dict) else {}
            seq = parse.get('text_sequence', '') if isinstance(parse, dict) else str(parse)
        except Exception:
            seq = gt if isinstance(gt, str) else ''
        return ' '.join(seq.split()[:MAX_WORDS])
    return 'sample document'


class DocumentDataset(Dataset):
    def __init__(self, dataset_name='nielsr/funsd', split='train', processor=None, max_samples=None, augment=False):
        self.processor = processor
        self.augment = augment
        raw = load_dataset(dataset_name, split=split)
        if max_samples and max_samples < len(raw):
            raw = raw.select(range(max_samples))
        self.dataset = raw

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        sample = self.dataset[idx]
        img = sample['image'].convert('RGB')
        if self.augment:
            img = DOC_AUG(img)
        pixel_values = self.processor(img, return_tensors='pt').pixel_values.squeeze(0)
        
        target_json = json.dumps({'text': build_target_text(sample)})
        prompt = f'<s_doc>{target_json}</s>'
        
        labels = self.processor.tokenizer(
            prompt, add_special_tokens=False, max_length=512, padding='max_length', truncation=True, return_tensors='pt'
        ).input_ids.squeeze(0)
        labels[labels == self.processor.tokenizer.pad_token_id] = -100
        
        decoder_input_ids = labels.clone()
        decoder_input_ids[decoder_input_ids == -100] = self.processor.tokenizer.pad_token_id
        return {
            'pixel_values': pixel_values,
            'labels': labels,
            'decoder_input_ids': decoder_input_ids,
            'raw_text': target_json
        }

def collate_fn(batch):
    return {
        'pixel_values': torch.stack([item['pixel_values'] for item in batch]),
        'labels': torch.stack([item['labels'] for item in batch]),
        'decoder_input_ids': torch.stack([item['decoder_input_ids'] for item in batch])
    }

processor = DonutProcessor.from_pretrained('naver-clova-ix/donut-base')

# Phase 2b: the SynthDoG-heavy mix (run 4: 63% synth, encoder frozen) REGRESSED
# recall -- clean synthetic text is off-domain for FUNSD's noisy scans, and a
# frozen encoder cannot adapt to it anyway. Now that the top Swin stage is
# unfrozen (see training cell) the encoder can learn FUNSD glyphs directly, so
# SynthDoG is demoted to a small anti-overfit garnish and FUNSD dominates (~70%).
SYNTH_N, FUNSD_REPEAT = 500, 8

# In eval-only mode none of this is used: the training loop never runs, and
# train_loader is referenced ONLY inside the epoch loop body. Building it anyway
# cost ~19 minutes of SynthDoG download on the 2026-08-29 attempt before the run
# failed on an unrelated path error, so skip it outright.
if EVAL_ONLY:
    train_ds = train_loader = funsd_train = synth_train = None
    print('EVAL-ONLY mode: skipped train set (no FUNSD-aug / SynthDoG download)')
else:
    funsd_train = DocumentDataset(dataset_name='nielsr/funsd', split='train', processor=processor, augment=True)
    synth_train = DocumentDataset(dataset_name='naver-clova-ix/synthdog-en', split=f'train[:{SYNTH_N}]', processor=processor, augment=False)
    train_ds = ConcatDataset([funsd_train] * FUNSD_REPEAT + [synth_train])
    print(f'Mixed train set: {len(funsd_train)} FUNSD x{FUNSD_REPEAT} + {len(synth_train)} SynthDoG = {len(train_ds)} samples')
    # batch_size = 1 with gradient accumulation = 8 ensures no OOM
    train_loader = DataLoader(train_ds, batch_size=1, shuffle=True, collate_fn=collate_fn)
    print(f'Train samples: {len(train_ds)}')

test_ds = DocumentDataset(dataset_name='nielsr/funsd', split='test', processor=processor)
print(f'Test samples: {len(test_ds)}')

## 4. Training on Kaggle GPU
Trains the Saliency Router and fine-tunes token economy with mixed precision (`fp16`).

In [ ]:
# Cell 7: Model Training
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# Phase 1: pruning OFF (keep all tokens) to establish the accuracy ceiling.
model = AdaptiveDonutOCR(keep_ratio=1.0, merge_ratio=0.0, freeze_encoder=True).to(device)

# Phase 2b: unfreeze the top Swin stage (+ final encoder norm where present) so
# the visual features can ADAPT to FUNSD's degraded fax/scan glyphs -- the
# bottleneck that decoder-side data (runs 3-4) could not move. The frozen early
# stages emit non-grad outputs, so autograd builds no graph for them; only the
# unfrozen tail (on the small 4800-token final grid) costs memory. forward()'s
# encoder_is_frozen check then auto-runs the encoder WITH grad -- no model-class
# change required. NOTE: DonutSwinModel.layernorm exists on transformers 4.x
# (Kaggle) but was folded away by 5.x, so unfreeze it defensively via getattr.
UNFREEZE_STAGES = 1
_swin_stages = model.model.encoder.encoder.layers
assert len(_swin_stages) >= UNFREEZE_STAGES, f'unexpected Swin structure: {len(_swin_stages)} stages'
for _stage in _swin_stages[-UNFREEZE_STAGES:]:
    for _p in _stage.parameters():
        _p.requires_grad = True
_final_norm = getattr(model.model.encoder, 'layernorm', None)
if _final_norm is not None:
    for _p in _final_norm.parameters():
        _p.requires_grad = True
_enc_trainable = sum(p.numel() for p in model.model.encoder.parameters() if p.requires_grad)
print(f'Unfroze top {UNFREEZE_STAGES} Swin stage(s){" + final norm" if _final_norm is not None else ""}: {_enc_trainable/1e6:.1f}M encoder params trainable')

# Train Router, Text Decoder, and the newly-unfrozen Swin tail. Discriminative
# LRs: router (fresh head) fastest; decoder mid; encoder slowest (1e-5) since
# pretrained Swin features are delicate and a high LR would wreck them.
_enc_trainable_params = [p for p in model.model.encoder.parameters() if p.requires_grad]
trainable_params = [
    {'params': model.router.parameters(), 'lr': 1e-4},
    {'params': model.model.decoder.parameters(), 'lr': 2e-5},
    {'params': _enc_trainable_params, 'lr': 1e-5}
]
optimizer = torch.optim.AdamW(trainable_params, weight_decay=0.01)

# lambda_sparsity=0.0 -> no token-budget pressure this phase (accuracy first)
criterion = AdaptivePruningLoss(lambda_sparsity=0.0, lambda_entropy=0.05, target_budget=1.0,
                            pad_token_id=processor.tokenizer.pad_token_id)
scaler = torch.amp.GradScaler('cuda' if device.type == 'cuda' else 'cpu')

EPOCHS = 5
GRAD_ACCUM = 8  # Effective batch size = 8

# Phase 2c: eval-only resume. RESUME_CKPT is set and validated in CELL 2 (before
# the dataset download), so by here the path is known to exist. EPOCHS=0 makes
# range(1, EPOCHS+1) empty, so the training loop body never executes -- no
# re-indentation of the loop required.
if RESUME_CKPT:
    _sd = torch.load(RESUME_CKPT, map_location=device, weights_only=True)
    _missing, _unexpected = model.load_state_dict(_sd, strict=False)
    assert not _unexpected, f'unexpected keys in checkpoint: {list(_unexpected)[:5]}'
    print(f'Loaded {RESUME_CKPT} (missing={len(_missing)}, unexpected={len(_unexpected)})')
    EPOCHS = 0
    print('EVAL-ONLY mode: training loop skipped (EPOCHS=0)')

print('Starting Training (partial encoder fine-tune; VRAM ~3-5 GB on T4)...')
for epoch in range(1, EPOCHS + 1):
    model.train()
    epoch_loss, epoch_ce = 0.0, 0.0
    optimizer.zero_grad()
    
    pbar = tqdm(train_loader, desc=f'Epoch {epoch}/{EPOCHS}')
    for step, batch in enumerate(pbar):
        pixel_values = batch['pixel_values'].to(device)
        labels = batch['labels'].to(device)
        decoder_input_ids = batch['decoder_input_ids'].to(device)
        
        with torch.amp.autocast('cuda' if device.type == 'cuda' else 'cpu'):
            outputs = model(pixel_values=pixel_values, labels=labels, decoder_input_ids=decoder_input_ids)
            loss_dict = criterion(outputs['logits'], labels, outputs['scores'], outputs['loss'])
            loss = loss_dict['loss'] / GRAD_ACCUM
            
        scaler.scale(loss).backward()
        if (step + 1) % GRAD_ACCUM == 0 or (step + 1) == len(train_loader):
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()
            
        epoch_loss += loss_dict['loss'].item()
        epoch_ce += loss_dict['ce_loss'].item()
        pbar.set_postfix({
            'Loss': f'{loss_dict["loss"].item():.3f}',
            'Tokens': f'{outputs["compressed_tokens"]}/{outputs["original_tokens"]}'
        })
        
    print(f'Epoch {epoch} Complete | Avg Loss: {epoch_loss/len(train_loader):.4f} | Avg CE: {epoch_ce/len(train_loader):.4f}')

if EPOCHS > 0:
    os.makedirs('/kaggle/working/checkpoints', exist_ok=True)
    torch.save(model.state_dict(), '/kaggle/working/checkpoints/adaptive_donut_funsd.pt')
    print('Saved model to /kaggle/working/checkpoints/adaptive_donut_funsd.pt')
else:
    print('EVAL-ONLY mode: skipped checkpoint save (weights identical to RESUME_CKPT)')

## 5. Benchmarking & Heatmap Visualizations
Calculates Normalized Edit Distance (NED), latency speedup, and visualizes pruned document regions.

In [ ]:
# Cell 8: Evaluation & Heatmap Plot
def compute_word_metrics(pred, gt_words):
    import re
    pw = re.findall(r'\w+', pred.lower())
    gold = re.findall(r'\w+', ' '.join(gt_words).lower())  # tokenize gold the same way
    gs = set(gold)
    recall = len(gs & set(pw)) / len(gs) if gs else 0.0
    try:
        import editdistance
        d = editdistance.eval(pw, gold)
    except Exception:
        d = abs(len(pw) - len(gold))
    order = max(0.0, 1.0 - d / max(len(pw), len(gold), 1))
    return recall, order


def compute_ned(pred, target):
    try:
        import editdistance
        d = editdistance.eval(pred, target)
    except Exception:
        d = sum(1 for a, b in zip(pred, target) if a != b) + abs(len(pred) - len(target))
    m = max(len(pred), len(target))
    return d / m if m > 0 else 0.0

model.eval()
test_raw = load_dataset('nielsr/funsd', split='test')  # all 50 test samples for stable metrics
neds, latencies, compression_ratios = [], [], []
recalls, word_orders = [], []
first_sample_meta = None

# Generation MUST be primed with the same task prompt the model was trained on
# (<s_doc>), otherwise it starts from the wrong token and produces junk.
TASK_PROMPT = '<s_doc>'
prompt_ids = processor.tokenizer(TASK_PROMPT, add_special_tokens=False, return_tensors='pt').input_ids.to(device)

for i, sample in enumerate(tqdm(test_raw, desc='Evaluating Test Set')):
    img = sample['image'].convert('RGB')
    pixel_values = processor(img, return_tensors='pt').pixel_values.to(device)
    words = sample.get('words', [])
    boxes = sample.get('bboxes') or sample.get('boxes')
    gt_words = reading_order_words(words, boxes)[:MAX_WORDS] if boxes else words[:MAX_WORDS]
    gt_str = json.dumps({'text': ' '.join(gt_words)})
    
    t0 = time.perf_counter()
    with torch.no_grad():
        gen_ids, meta = model.generate(pixel_values, decoder_input_ids=prompt_ids, max_length=512)
    latencies.append((time.perf_counter() - t0) * 1000.0)
    
    pred = processor.batch_decode(gen_ids, skip_special_tokens=True)[0]
    # Strip the echoed task prompt so we score only the generated content
    if pred.startswith(TASK_PROMPT):
        pred = pred[len(TASK_PROMPT):]
    pred = pred.strip()
    neds.append(compute_ned(pred, gt_str))
    rec, order = compute_word_metrics(pred, gt_words)
    recalls.append(rec)
    word_orders.append(order)
    compression_ratios.append(meta['compression_ratio'])
    if i == 0:
        first_sample_meta = (img, meta)
        # Print real text so success/failure is visible, not hidden behind a metric
        print('\n--- Sample 0 sanity check ---')
        print(f'PRED: {pred[:200]}')
        print(f'GT  : {gt_str[:200]}')
        print('-----------------------------')

print('\n' + '='*52)
print(f'[Stage 1] Word Recall (no missing words): {np.mean(recalls)*100:.2f}%')
print(f'[Stage 2] Character Accuracy:             {(1.0 - np.mean(neds))*100:.2f}%')
print(f'[Stage 3] Word-Order / Sequence Score:    {np.mean(word_orders)*100:.2f}%')
print(f'Mean Normalized Edit Distance (NED):      {np.mean(neds):.4f}')
print(f'Average Visual Token Savings:             {np.mean(compression_ratios):.1f}%')
print(f'Average Inference Latency:                {np.mean(latencies):.1f} ms/doc')
print('='*52)

# Visualize Overlay on First Sample
sample_img, meta = first_sample_meta
img_np = np.array(sample_img)
H, W, _ = img_np.shape
N = meta['original_tokens']
W_grid = int(round((N * (4/3)) ** 0.5))
H_grid = max(1, N // W_grid)
mask = np.zeros((H_grid, W_grid))
selected = set(meta['topk_indices'][0].cpu().numpy().tolist())
for idx in range(min(H_grid * W_grid, N)):
    if idx in selected:
        mask[idx // W_grid, idx % W_grid] = 1.0

mask_res = np.array(Image.fromarray((mask * 255).astype(np.uint8)).resize((W, H), Image.NEAREST)) / 255.0
overlay = img_np.copy().astype(np.float32)
overlay[mask_res < 0.5] *= 0.35

fig, axes = plt.subplots(1, 2, figsize=(12, 6))
axes[0].imshow(img_np)
axes[0].set_title('Original Document Image')
axes[0].axis('off')
axes[1].imshow(overlay.astype(np.uint8))
axes[1].set_title(f'Retained Tokens ({meta["compressed_tokens"]}/{N} - {meta["compression_ratio"]:.0f}% Savings)')
axes[1].axis('off')
plt.tight_layout()
os.makedirs('/kaggle/working', exist_ok=True)
plt.savefig('/kaggle/working/token_pruning_overlay.png', dpi=200)
plt.show()

In [ ]:
# Cell 8b: Decoding ablation (EVAL-ONLY -- no retraining)
# Tests whether the stale generation config, not the model, caps word recall.
# repetition_penalty=1.3 + no_repeat_ngram_size=3 were tuned for AGGRESSIVE
# pruning ("repetition collapse"), but runs 2-5 all run keep_ratio=1.0 (pruning
# OFF). On full-page forms, trigram blocking is actively harmful: real documents
# repeat trigrams (form labels, phone numbers, dotted leaders), and once every
# continuation is blocked EOS becomes top-1, so the model stops mid-document.
# Re-evaluates the SAME weights under several decoding configs.
import re

# Row 0 is a CONTROL: it uses run 5's exact settings and must reproduce run 5's
# numbers. If it does not, this harness differs from the eval cell and NO other
# row is trustworthy -- the drift check below prints that verdict.
RUN5_REFERENCE = (50.56, 40.97, 26.49)  # recall / charAcc / order, FUNSD test x50
ABLATION_MAX_LEN = 512

DECODE_CONFIGS = [
    ('baseline (run5)',   dict(repetition_penalty=1.3, no_repeat_ngram_size=3)),
    ('no ngram block',    dict(repetition_penalty=1.3, no_repeat_ngram_size=0)),
    ('no rep penalty',    dict(repetition_penalty=1.0, no_repeat_ngram_size=3)),
    ('both off',          dict(repetition_penalty=1.0, no_repeat_ngram_size=0)),
    ('both off + minlen', dict(repetition_penalty=1.0, no_repeat_ngram_size=0, min_new_tokens=64)),
]


def run_decode_eval(label, overrides):
    """Re-run the FUNSD test set under one decoding config.

    Metrics mirror the eval cell exactly (same compute_word_metrics / compute_ned
    / reading_order_words / MAX_WORDS) so rows stay comparable to runs 2-5. The
    added length fields test the under-generation hypothesis directly rather than
    inferring it from recall.
    """
    model.eval()
    recs, ords_, neds_ = [], [], []
    pred_words, gold_words, gen_tokens = [], [], []
    json_ok, hit_cap = 0, 0

    for sample in tqdm(test_raw, desc=label, leave=False):
        img = sample['image'].convert('RGB')
        pv = processor(img, return_tensors='pt').pixel_values.to(device)
        words = sample.get('words', [])
        boxes = sample.get('bboxes') or sample.get('boxes')
        gt_words = reading_order_words(words, boxes)[:MAX_WORDS] if boxes else words[:MAX_WORDS]
        gt_str = json.dumps({'text': ' '.join(gt_words)})

        with torch.no_grad():
            gen_ids, _meta = model.generate(
                pv, decoder_input_ids=prompt_ids,
                max_length=ABLATION_MAX_LEN, **overrides
            )

        n_tok = int(gen_ids.shape[-1])
        gen_tokens.append(n_tok)
        if n_tok >= ABLATION_MAX_LEN:
            hit_cap += 1   # ran to the cap instead of emitting EOS

        pred = processor.batch_decode(gen_ids, skip_special_tokens=True)[0]
        if pred.startswith(TASK_PROMPT):
            pred = pred[len(TASK_PROMPT):]
        pred = pred.strip()

        # diagnostic only -- we never repair pred, that would score the fix
        try:
            json.loads(pred)
            json_ok += 1
        except Exception:
            pass

        r, o = compute_word_metrics(pred, gt_words)
        recs.append(r)
        ords_.append(o)
        neds_.append(compute_ned(pred, gt_str))
        pred_words.append(len(re.findall(r'\w+', pred.lower())))
        gold_words.append(len(re.findall(r'\w+', ' '.join(gt_words).lower())))

    n = max(len(recs), 1)
    mp, mg = float(np.mean(pred_words)), float(np.mean(gold_words))
    return {
        'config': label,
        'overrides': {k: v for k, v in overrides.items()},
        'word_recall_pct': float(np.mean(recs) * 100.0),
        'character_accuracy_pct': float((1.0 - np.mean(neds_)) * 100.0),
        'word_order_pct': float(np.mean(ords_) * 100.0),
        'mean_ned': float(np.mean(neds_)),
        'mean_pred_words': mp,
        'mean_gold_words': mg,
        'len_ratio_pct': 100.0 * mp / max(mg, 1e-9),
        'mean_gen_tokens': float(np.mean(gen_tokens)),
        'hit_max_length_pct': 100.0 * hit_cap / n,
        'valid_json_pct': 100.0 * json_ok / n,
        'num_eval_samples': len(recs),
    }


rows = []
for _name, _cfg in DECODE_CONFIGS:
    print(f'--- decoding: {_name}  {_cfg}')
    rows.append(run_decode_eval(_name, _cfg))

hdr = (f"{'config':20s} {'recall':>7s} {'charAcc':>8s} {'order':>7s} {'NED':>6s} "
       f"{'predW':>6s} {'goldW':>6s} {'len%':>6s} {'cap%':>5s} {'json%':>6s}")
print('\n' + '=' * len(hdr))
print(hdr)
print('-' * len(hdr))
for r in rows:
    print(f"{r['config']:20s} {r['word_recall_pct']:7.2f} {r['character_accuracy_pct']:8.2f} "
          f"{r['word_order_pct']:7.2f} {r['mean_ned']:6.3f} {r['mean_pred_words']:6.1f} "
          f"{r['mean_gold_words']:6.1f} {r['len_ratio_pct']:6.1f} "
          f"{r['hit_max_length_pct']:5.0f} {r['valid_json_pct']:6.1f}")
print('=' * len(hdr))

# --- CONTROL: does the baseline row reproduce run 5? ---
base = rows[0]
got = (base['word_recall_pct'], base['character_accuracy_pct'], base['word_order_pct'])
drift = max(abs(g - e) for g, e in zip(got, RUN5_REFERENCE))
print(f"\nCONTROL vs run 5 {RUN5_REFERENCE}")
print(f"  baseline got  ({got[0]:.2f}, {got[1]:.2f}, {got[2]:.2f})   max drift {drift:.2f} pts")
if drift < 0.5:
    print('  OK - harness reproduces run 5, so the other rows are comparable.')
else:
    print('  WARNING - harness does NOT match the run-5 eval (different weights, '
          'sampling, or transformers version). Treat other rows as suspect.')

# --- did loosening the decoder actually lengthen the output? ---
best = max(rows, key=lambda r: r['word_recall_pct'])
print(f"\nBest recall: '{best['config']}' at {best['word_recall_pct']:.2f}% "
      f"({best['word_recall_pct'] - base['word_recall_pct']:+.2f} vs baseline)")
print(f"  words emitted: {best['mean_pred_words']:.1f} vs baseline "
      f"{base['mean_pred_words']:.1f} (gold {base['mean_gold_words']:.1f})")
if best['config'] != base['config'] and best['mean_pred_words'] > base['mean_pred_words']:
    print('  => recall rose TOGETHER WITH length: under-generation was the cap.')
    print('     Re-run the Phase 2 pruning sweep with this config, not the stale one.')
elif best['config'] == base['config']:
    print('  => no config beat the baseline: the cap is NOT decoding. Look at the')
    print('     128-word target cap / encoder capacity instead (UNFREEZE_STAGES=2).')
else:
    print('  => recall rose WITHOUT more words: gain is fidelity, not coverage.')

with open('/kaggle/working/ablation_decoding.json', 'w') as f:
    json.dump(rows, f, indent=2)
print('\nWrote /kaggle/working/ablation_decoding.json')


In [ ]:
# Cell 8c: SELECTION ablation (EVAL-ONLY -- no retraining)
# Answers Pending 1b / 1c / 1d from AGENTS.md in one GPU session. Same weights, same
# decoder, same token BUDGET within a keep_ratio -- only the RANKING SIGNAL moves, so
# rows are comparable at equal cost.
#
# Two controls run at keep_ratio=1.0 (pruning OFF), and they are not the same check:
#   HARNESS CONTROL loads the run-5 weights that PRODUCED RUN6_REFERENCE and must
#     reproduce it. Nothing varies vs run 6, so drift here is the measurement --
#     harness, library version or sampling -- and nothing else is trustworthy.
#   Row 0 uses the weights actually being swept. Once the harness is verified, its
#     drift from run 6 is what the retrain did to unpruned accuracy.
# Runs 7-8 had only row 0, which retraining had turned into a two-variable test: its
# drift could not distinguish a broken harness from a changed checkpoint.
#
# 1b  keep=0.50 x {router, negated, random, ink}
#       random is the FLOOR: D1 measured the trained router as anti-correlated with
#       patch ink (r ~ -0.24), so uniform noise retains more text than the learned
#       scores. A router that cannot beat noise is not a router.
#       ink is the CEILING: patch contrast, no learning. It bounds what any
#       ink-seeking router could reach on this decoder. If it collapses, the premise
#       that half these tokens are droppable is wrong and Pending 1a is 4 wasted
#       GPU-hours. (Local CPU n=4: oracle 83.66 vs full page 83.67 -- premise holds.)
# 1d  keep=0.50 x {stratified, stratified_negated}
#       D2: the router's MINIMUM text-line ink coverage is 0.000 -- at least one line
#       loses all its ink -- while total retained ink looks unremarkable, and recall is
#       gated by the worst line. A per-grid-row budget makes that impossible. If these
#       rows fix the router, the bug is the SELECTION RULE, not the scorer, and
#       retraining is the wrong fix.
# 1c  keep in {1.00, 0.75, 0.50, 0.35}, each with its floor and ceiling
#       The accuracy/cost CURVE, with latency and token counts per row. GW=60 makes
#       the per-row budget integral at every ratio (60/45/30/21), so the stratified
#       rows are exact per-row top-k rather than the degradation path.
import os
import re
import time

RUN6_REFERENCE = (77.74, 64.70, 53.05)   # recall / charAcc / order, FUNSD test x50
SELECTION_MAX_LEN = 512
SELECTION_SEED = 0
GH, GW = 80, 60          # Swin-B token grid at 2560x1920, stride 32
ROW_INK_FRAC = 0.10     # a grid row counts as text if its ink >= 10% of the busiest
SELECTION_OUT = os.path.join(
    '/kaggle/working' if os.path.isdir('/kaggle/working') else '.',
    'ablation_selection.json')

SELECTION_CONFIGS = [
    ('keep=1.00 router CONTROL', 1.00, 'router'),
    # 1b
    ('keep=0.50 router',         0.50, 'router'),
    ('keep=0.50 NEGATED',        0.50, 'negated'),
    ('keep=0.50 random',         0.50, 'random'),
    ('keep=0.50 ink ORACLE',     0.50, 'ink'),
    # 1d
    ('keep=0.50 stratified',     0.50, 'stratified'),
    ('keep=0.50 strat NEGATED',  0.50, 'stratified_negated'),
    # 1c
    ('keep=0.75 router',         0.75, 'router'),
    ('keep=0.75 random',         0.75, 'random'),
    ('keep=0.75 ink ORACLE',     0.75, 'ink'),
    ('keep=0.75 strat NEGATED',  0.75, 'stratified_negated'),
    ('keep=0.35 router',         0.35, 'router'),
    ('keep=0.35 random',         0.35, 'random'),
    ('keep=0.35 ink ORACLE',     0.35, 'ink'),
    ('keep=0.35 strat NEGATED',  0.35, 'stratified_negated'),
]


def line_coverage(pv, topk_indices):
    """Per-text-row retained-ink fractions for one image -> 1-D array, or None.

    D2 found that summed retained ink ranks `negated` ABOVE `random` while accuracy
    does the opposite, and that the WORST-covered text line is what tracks recall.
    That was 4 cached images against 4 aggregate rows -- one statistic ordering 4 rows
    correctly is 1-in-24 by luck. Recorded per image here so it becomes a real
    correlation that can actually fail.
    """
    N = GH * GW
    if pv.shape[-2] // 32 != GH or pv.shape[-1] // 32 != GW:
        return None            # non-standard grid; the accuracy row is still valid
    ink = patch_ink(pv, N)[0].reshape(GH, GW).float().cpu().numpy()
    keep = np.zeros(N, dtype=bool)
    keep[topk_indices[0].cpu().numpy()] = True
    keep2d = keep.reshape(GH, GW)
    row_ink = ink.sum(axis=1)
    is_text = row_ink >= ROW_INK_FRAC * row_ink.max()
    if not is_text.any():
        return None
    return (ink * keep2d).sum(axis=1)[is_text] / np.maximum(row_ink[is_text], 1e-9)


def run_selection_eval(label, keep_ratio, select_mode):
    """Re-run the FUNSD test set under one (keep_ratio, ranking signal) pair.

    Metrics mirror the eval cell exactly (same compute_word_metrics / compute_ned /
    reading_order_words / MAX_WORDS) so rows stay comparable to runs 2-6.
    """
    model.eval()
    # Reseed per row so the random control draws the same sequence on every run while
    # still getting a FRESH mask per image.
    torch.manual_seed(SELECTION_SEED)
    recs, ords_, neds_, inks, lats = [], [], [], [], []
    pred_words, gold_words, gen_tokens = [], [], []
    json_ok, hit_cap = 0, 0
    kept_tokens = None
    per_image = []

    for i, sample in enumerate(tqdm(test_raw, desc=label, leave=False)):
        img = sample['image'].convert('RGB')
        pv = processor(img, return_tensors='pt').pixel_values.to(device)
        words = sample.get('words', [])
        boxes = sample.get('bboxes') or sample.get('boxes')
        gt_words = reading_order_words(words, boxes)[:MAX_WORDS] if boxes else words[:MAX_WORDS]
        gt_str = json.dumps({'text': ' '.join(gt_words)})

        t0 = time.perf_counter()
        with torch.no_grad():
            gen_ids, meta = model.generate(
                pv, decoder_input_ids=prompt_ids, keep_ratio=keep_ratio,
                merge_ratio=0.0, max_length=SELECTION_MAX_LEN, select_mode=select_mode
            )
        lats.append((time.perf_counter() - t0) * 1000.0)
        kept_tokens = meta['compressed_tokens']
        if meta['retained_ink'] is not None:
            inks.append(meta['retained_ink'])
        cov = line_coverage(pv, meta['topk_indices'])

        n_tok = int(gen_ids.shape[-1])
        gen_tokens.append(n_tok)
        if n_tok >= SELECTION_MAX_LEN:
            hit_cap += 1   # ran to the cap instead of emitting EOS

        pred = processor.batch_decode(gen_ids, skip_special_tokens=True)[0]
        if pred.startswith(TASK_PROMPT):
            pred = pred[len(TASK_PROMPT):]
        pred = pred.strip()

        # diagnostic only -- we never repair pred, that would score the fix
        try:
            json.loads(pred)
            json_ok += 1
        except Exception:
            pass

        r, o = compute_word_metrics(pred, gt_words)
        recs.append(r)
        ords_.append(o)
        neds_.append(compute_ned(pred, gt_str))
        pred_words.append(len(re.findall(r'\w+', pred.lower())))
        gold_words.append(len(re.findall(r'\w+', ' '.join(gt_words).lower())))
        per_image.append({
            'i': i,
            'recall': float(r),
            'ned': float(neds_[-1]),
            'retained_ink': meta['retained_ink'],
            'gen_tokens': n_tok,
            'min_line_cov': None if cov is None else float(cov.min()),
            'p10_line_cov': None if cov is None else float(np.percentile(cov, 10)),
            'mean_line_cov': None if cov is None else float(cov.mean()),
            'n_text_rows': None if cov is None else int(cov.size),
        })

    n = max(len(recs), 1)
    mp, mg = float(np.mean(pred_words)), float(np.mean(gold_words))
    covs = [p['min_line_cov'] for p in per_image if p['min_line_cov'] is not None]
    return {
        'config': label,
        'keep_ratio': keep_ratio,
        'select_mode': select_mode,
        'visual_tokens': kept_tokens,
        'retained_ink': float(np.mean(inks)) if inks else None,
        'mean_min_line_cov': float(np.mean(covs)) if covs else None,
        'word_recall_pct': float(np.mean(recs) * 100.0),
        'character_accuracy_pct': float((1.0 - np.mean(neds_)) * 100.0),
        'word_order_pct': float(np.mean(ords_) * 100.0),
        'mean_ned': float(np.mean(neds_)),
        'mean_pred_words': mp,
        'mean_gold_words': mg,
        'len_ratio_pct': 100.0 * mp / max(mg, 1e-9),
        'mean_gen_tokens': float(np.mean(gen_tokens)),
        'hit_max_length_pct': 100.0 * hit_cap / n,
        'valid_json_pct': 100.0 * json_ok / n,
        'avg_latency_ms': float(np.mean(lats)),
        'num_eval_samples': len(recs),
        'per_image': per_image,
    }


_t0 = time.perf_counter()

# ---------------------------------------------------------------- HARNESS CONTROL
# The row-0 CONTROL below re-runs a known SETTING; this block re-runs a known
# CHECKPOINT. Runs 7-8 retrained before row 0 ran, so row 0's drift from run 6
# conflated "the harness moved" with "the weights moved". Evaluating the run-5
# weights -- the ones that produced RUN6_REFERENCE -- through this exact function
# varies nothing vs run 6, so any drift HERE is harness/version/sampling, and row
# 0's drift is then the retrain alone. keep_ratio=1.0 prunes nothing, so this row
# is also independent of every selection code path under test.
HARNESS_CONTROL_TOL = 0.5
_ctrl_ckpt = globals().get('RESUME_CKPT') or None
_eval_ckpt = globals().get('EVAL_CKPT') or _ctrl_ckpt
_retrained = bool(globals().get('DO_TRAIN', False))
harness_row, harness_verified, harness_note = None, None, ''


def _weight_fingerprint(m):
    """Deterministic checksum over EVERY parameter, not a sample of them.

    A sample can miss the swap: the first/middle/last keys of a Donut state dict are
    frozen encoder tensors that training never touches, so a 3-key fingerprint would
    compare equal across two genuinely different checkpoints. Cost is one .sum() per
    tensor, milliseconds on GPU, and only precision-stable equality is needed here.
    """
    return tuple(float(v.detach().float().sum()) for _, v in sorted(m.state_dict().items()))


if not _retrained:
    harness_note = ('skipped: DO_TRAIN is off, so `model` still holds the control weights '
                    'and row 0 below already IS the harness control.')
    print(f'\nHARNESS CONTROL {harness_note}')
elif not (_ctrl_ckpt and os.path.exists(_ctrl_ckpt)):
    harness_note = f'skipped: control checkpoint not found ({_ctrl_ckpt!r})'
    print(f'\nHARNESS CONTROL {harness_note}')
else:
    print(f'\n--- HARNESS CONTROL: run-5 weights through this harness')
    print(f'    control  {_ctrl_ckpt}')
    print(f'    restore  {_eval_ckpt}')
    _fp_swept = _weight_fingerprint(model)
    _sd = torch.load(_ctrl_ckpt, map_location='cpu', weights_only=True)
    _miss, _unex = model.load_state_dict(_sd, strict=False)
    assert not _miss and not _unex, (
        f'control checkpoint does not match this architecture: {len(_miss)} missing, '
        f'{len(_unex)} unexpected (missing {_miss[:3]}, unexpected {_unex[:3]}). A partial '
        f'load would evaluate a partly-random model and call the drift a control.')
    assert _weight_fingerprint(model) != _fp_swept, (
        'control load changed nothing -- RESUME_CKPT and the swept weights are the same '
        'tensors, so this row would not be a control.')

    harness_row = run_selection_eval('HARNESS CONTROL run-5 wts', 1.00, 'router')

    # Restore BEFORE reporting, so a restore failure cannot be mistaken for a
    # reporting failure and so the assert below fires while the cause is on screen.
    _sd2 = torch.load(_eval_ckpt, map_location='cpu', weights_only=True)
    _m2, _u2 = model.load_state_dict(_sd2, strict=False)
    assert not _m2 and not _u2, (
        f'restore checkpoint does not match: {len(_m2)} missing, {len(_u2)} unexpected')
    assert _weight_fingerprint(model) == _fp_swept, (
        'FAILED TO RESTORE the swept weights. Every row below would measure the control '
        'checkpoint instead, and the table would look entirely plausible. Refusing to '
        'produce it.')
    del _sd, _sd2
    print('    weights restored and fingerprint-verified')

    _hgot = (harness_row['word_recall_pct'], harness_row['character_accuracy_pct'],
             harness_row['word_order_pct'])
    _hdrift = max(abs(g - e) for g, e in zip(_hgot, RUN6_REFERENCE))
    harness_verified = bool(_hdrift < HARNESS_CONTROL_TOL)
    print(f'    got ({_hgot[0]:.2f}, {_hgot[1]:.2f}, {_hgot[2]:.2f})  vs run 6 '
          f'{RUN6_REFERENCE}  max drift {_hdrift:.2f} pts')
    if harness_verified:
        harness_note = f'verified: run-5 weights reproduce run 6 within {_hdrift:.2f} pts'
        print('    HARNESS OK - the same weights reproduce run 6 through this code, so '
              'row 0\'s')
        print('    drift below is the RETRAIN, not the measurement.')
    else:
        harness_note = (f'FAILED: run-5 weights drift {_hdrift:.2f} pts from run 6 through '
                        f'this harness')
        print('    ' + '!' * 70)
        print('    HARNESS CONTROL FAILED. The weights that produced run 6 do NOT')
        print('    reproduce it here, so nothing below is quotable against run 6 or any')
        print('    earlier run in absolute terms. Rows stay comparable TO EACH OTHER')
        print('    (same harness, same weights, only the ranking signal varies), which')
        print('    is what Q1/Q2/Q4 ask -- so the sweep continues instead of discarding')
        print('    the session. Results are stamped harness_verified=false.')
        print('    ' + '!' * 70)

rows = []

# ------------------------------------------------------------------- PROVENANCE
# What produced the rows below, stamped into the JSON at BOTH write sites. The
# settings of a run were already recorded; the ENVIRONMENT was not, and that is the
# axis every cross-run comparison in this project has broken on (D11/D12: local
# generation drifts up to 3.4 pts from Kaggle on bit-identical selections, different
# device and transformers version). mtime+size as well as the path, because run
# directories get renamed and a bare path stops identifying the file.
#
# Nothing here may raise: provenance is bookkeeping, and bookkeeping must not be able
# to cost a 40-minute GPU session. Every lookup degrades to a string instead.
try:
    import transformers as _tfm_prov
    _tfm_ver = _tfm_prov.__version__
except Exception as _e:                       # noqa: BLE001 - report, never raise
    _tfm_ver = f'unavailable ({type(_e).__name__})'


def _ckpt_stamp(p):
    """Path + mtime + size for a checkpoint, or an explicit reason there is none.

    Returns a dict on every path, including failure. A MISSING key in the JSON means
    "written before provenance existed"; a PRESENT key carrying 'error' means "we
    looked and it was not there". Collapsing both to None discards exactly the
    distinction that matters when reading a stale artifact months later.
    """
    if not p:
        return {'path': None,
                'error': 'no checkpoint variable set (EVAL_CKPT / RESUME_CKPT)'}
    try:
        _st = os.stat(p)
    except OSError as e:
        return {'path': str(p), 'error': f'{type(e).__name__}: {e}'}
    return {'path': str(p),
            'mtime': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime(_st.st_mtime)),
            'size_bytes': int(_st.st_size)}


try:
    _dev_prov = str(next(model.parameters()).device)
except Exception:                             # noqa: BLE001 - NameError / StopIteration
    _dev_prov = 'unknown'
if _dev_prov.startswith('cuda'):
    try:
        _devname_prov = torch.cuda.get_device_name(_dev_prov)
    except Exception:                         # noqa: BLE001
        _devname_prov = 'cuda (name unavailable)'
elif _dev_prov == 'unknown':
    _devname_prov = 'unknown'
else:
    _devname_prov = 'cpu'

PROVENANCE = {
    'eval_ckpt': _ckpt_stamp(_eval_ckpt),
    'control_ckpt': _ckpt_stamp(_ctrl_ckpt),
    'transformers': _tfm_ver,
    'torch': getattr(torch, '__version__', 'unknown'),
    'device': _dev_prov,
    'device_name': _devname_prov,
    'written': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()),
}
# Serialise once HERE rather than discovering at the first write that something in the
# dict is not JSON-able -- that write lands after row 0, i.e. minutes of GPU in.
json.dumps(PROVENANCE)
print(f"\nPROVENANCE  torch {PROVENANCE['torch']}  transformers "
      f"{PROVENANCE['transformers']}  {PROVENANCE['device']} ({PROVENANCE['device_name']})")
print(f"    eval ckpt     {PROVENANCE['eval_ckpt']}")
print(f"    control ckpt  {PROVENANCE['control_ckpt']}")

for _name, _kr, _mode in SELECTION_CONFIGS:
    print(f'--- {_name}')
    rows.append(run_selection_eval(_name, _kr, _mode))
    _r = rows[-1]
    _ink = '  -  ' if _r['retained_ink'] is None else f"{_r['retained_ink']:.3f}"
    print(f"    -> recall {_r['word_recall_pct']:.2f}  charAcc "
          f"{_r['character_accuracy_pct']:.2f}  ink {_ink}  "
          f"{_r['avg_latency_ms']:.0f} ms/img  [{(time.perf_counter() - _t0) / 60:.1f} min]")
    # Written after EVERY row: a Kaggle session that dies at row 12 must not lose
    # rows 0-11 the way the local run did.
    with open(SELECTION_OUT, 'w') as f:
        json.dump({'meta': {'run6_reference': RUN6_REFERENCE,
                            'seed': SELECTION_SEED,
                            'max_len': SELECTION_MAX_LEN,
                            'harness_verified': harness_verified,
                            'harness_note': harness_note,
                            'harness_control': harness_row,
                            'provenance': PROVENANCE,
                            'complete': False}, 'rows': rows}, f, indent=2)

hdr = (f"{'config':26s} {'tok':>5s} {'ink':>5s} {'minCov':>6s} {'recall':>7s} "
       f"{'charAcc':>8s} {'order':>7s} {'NED':>6s} {'len%':>6s} {'cap%':>5s} "
       f"{'json%':>6s} {'ms':>7s}")
print('\n' + '=' * len(hdr))
print(hdr)
print('-' * len(hdr))
for r in rows:
    ink = '  -  ' if r['retained_ink'] is None else f"{r['retained_ink']:.3f}"
    mc = '  -  ' if r['mean_min_line_cov'] is None else f"{r['mean_min_line_cov']:.3f}"
    print(f"{r['config']:26s} {r['visual_tokens']:5d} {ink:>5s} {mc:>6s} "
          f"{r['word_recall_pct']:7.2f} {r['character_accuracy_pct']:8.2f} "
          f"{r['word_order_pct']:7.2f} {r['mean_ned']:6.3f} {r['len_ratio_pct']:6.1f} "
          f"{r['hit_max_length_pct']:5.0f} {r['valid_json_pct']:6.1f} "
          f"{r['avg_latency_ms']:7.0f}")
print('=' * len(hdr))

# --- CONTROL: does keep_ratio=1.0 on the SWEPT weights reproduce run 6? ---
# Read this together with the HARNESS CONTROL above. That block fixed the weights and
# varied nothing; this one fixes the setting and varies the weights. The pair is what
# makes a drift attributable: harness verified + row 0 drifting = the retrain moved the
# ceiling. Both drifting = measurement problem, and the retrain's effect is unreadable.
ctrl = rows[0]
got = (ctrl['word_recall_pct'], ctrl['character_accuracy_pct'], ctrl['word_order_pct'])
drift = max(abs(g - e) for g, e in zip(got, RUN6_REFERENCE))
print(f'\nCONTROL vs run 6 {RUN6_REFERENCE}')
print(f'  got ({got[0]:.2f}, {got[1]:.2f}, {got[2]:.2f})   max drift {drift:.2f} pts')
if drift < 0.5:
    print('  OK - this checkpoint matches run 6 at keep=1.0, so every other row is '
          'comparable.')
elif harness_verified:
    print(f'  DIFFERENT CEILING - and the harness was verified above, so this '
          f'{drift:.2f} pt gap')
    print('  is what the retrain did to unpruned accuracy, not measurement noise. Compare '
          'rows to THIS row, not to run 6.')
else:
    print('  WARNING - harness does NOT match run 6 (different weights, sampling, or '
          'transformers version). Rows are still comparable TO EACH OTHER and to this '
          'control, but do not quote them against run 6 absolutely.')
    if harness_verified is False:
        print('  The HARNESS CONTROL above also failed, so this gap cannot be attributed '
              'to the retrain.')

by = {(r['keep_ratio'], r['select_mode']): r for r in rows}


def rec(kr, mode):
    r = by.get((kr, mode))
    return None if r is None else r['word_recall_pct']


# --- Q1 (1b): does negating the router help? ---
print('\nQ1. Does negating the router help? (keep=0.50)')
for m in ('router', 'negated', 'random'):
    r = by[(0.50, m)]
    print(f"  {m:8s} {r['word_recall_pct']:6.2f} recall   ink {r['retained_ink']:.3f}"
          f"   minCov {r['mean_min_line_cov']:.3f}")
if rec(0.50, 'negated') > max(rec(0.50, 'router'), rec(0.50, 'random')):
    print('  => NEGATION IS A REAL FIX: it beats the forward router AND random, so '
          'flipping the sign makes the existing checkpoint usable.')
else:
    print('  => negation does not beat random: retaining ink is NECESSARY BUT NOT '
          'SUFFICIENT (D2). Do not adopt it as the fix.')

# --- Q2 (1b): is the premise sound at all? ---
print('\nQ2. Are half these tokens droppable at all? (the row that can kill 1a)')
print(f"  keep=1.00 CONTROL  {rec(1.00, 'router'):6.2f} recall  ({ctrl['visual_tokens']} tokens)")
_ik = by[(0.50, 'ink')]
print(f"  keep=0.50 ORACLE   {_ik['word_recall_pct']:6.2f} recall  "
      f"({_ik['visual_tokens']} tokens, ink {_ik['retained_ink']:.3f})   "
      f"{_ik['word_recall_pct'] - ctrl['word_recall_pct']:+.2f} vs control")
if _ik['word_recall_pct'] > ctrl['word_recall_pct'] - 5.0:
    print('  => PREMISE HOLDS: an ink-ranked half of the tokens nearly matches the '
          'full page, so the headroom a trained router could reach is real.')
else:
    print('  => PREMISE IS WEAK: even a perfect ink ranking loses accuracy at this '
          'budget. Cap expectations for ANY router and consider merge_ratio (which '
          'compresses rather than discards) before more pruning work.')
print(f"  at keep=0.50 the router sits at {rec(0.50, 'router'):.2f}, between a floor of "
      f"{rec(0.50, 'random'):.2f} (random) and a ceiling of {_ik['word_recall_pct']:.2f} (oracle).")

# --- Q4 (1d): is the SELECTION RULE the bug, or the scorer? ---
print('\nQ4. Is the bug the selection RULE or the scorer? (keep=0.50, D2/Pending 1d)')
for m in ('router', 'stratified', 'stratified_negated', 'random', 'ink'):
    r = by[(0.50, m)]
    print(f"  {m:19s} {r['word_recall_pct']:6.2f} recall   ink {r['retained_ink']:.3f}"
          f"   minCov {r['mean_min_line_cov']:.3f}")
_best_strat = max(rec(0.50, 'stratified'), rec(0.50, 'stratified_negated'))
if _best_strat >= rec(0.50, 'random'):
    print('  => THE SELECTION RULE WAS THE BUG. A per-row budget on the SAME scores '
          'reaches the random floor or better, so the scorer is salvageable and 1a '
          'should train WITH stratified selection rather than replace the scorer.')
elif _best_strat > rec(0.50, 'router') + 2.0:
    print('  => PARTIAL: stratification helps but does not reach random, so the rule '
          'is one cause and the scorer is another. 1a still needed; keep the per-row '
          'budget in it.')
else:
    print('  => NOT THE RULE. Bounding worst-case coverage does not rescue these '
          'scores, so the scorer itself is what is broken -- Pending 1a (train with '
          'pruning ON) is the fix, and D2 says its loss must NOT reward summed '
          'retained saliency.')

# --- Q5 (1c): the accuracy/cost curve ---
print('\nQ5. keep_ratio curve (Pending 1c) -- recall by ranking signal, cost per row')
print(f"  {'keep':>5s} {'tok':>5s} {'router':>7s} {'random':>7s} {'oracle':>7s} "
      f"{'stratN':>7s} {'ms/img':>7s}")
for kr in (1.00, 0.75, 0.50, 0.35):
    r0 = by.get((kr, 'router'))
    if r0 is None:
        continue
    cells_ = []
    for m in ('router', 'random', 'ink', 'stratified_negated'):
        v = rec(kr, m)
        cells_.append('      -' if v is None else f'{v:7.2f}')
    print(f"  {kr:5.2f} {r0['visual_tokens']:5d} " + ' '.join(cells_) +
          f" {r0['avg_latency_ms']:7.0f}")
print('  NOTE: there is NO encoder-side saving. The router runs AFTER the frozen')
print('        Swin, so all 4800 tokens are computed at every keep_ratio; the only')
print('        saving is decoder cross-attention KV length.')
print('  NOTE: latency is decoder-bound. D11 (n=50, 5 budgets, 2 checkpoints)')
print('        measured 4800 -> 960 tokens buying 1.04x wall-clock (1.05x max),')
print('        because generation runs 243-280 autoregressive steps while the')
print('        encoder runs once. The tok column is a TOKEN-COUNT axis, not an')
print('        efficiency one; any latency or throughput claim is false.')

# --- Q3 (D2): which statistic predicts recall? ---
# D2 claimed WORST-CASE line coverage discriminates where summed retained ink does
# not, but measured it on 4 cached images against 4 aggregate rows. Here it is a
# per-image test over every equal-budget row, so it can actually fail.
def spearman(a, b):
    a, b = np.asarray(a, dtype=float), np.asarray(b, dtype=float)
    if a.size < 3 or np.ptp(a) == 0 or np.ptp(b) == 0:
        return float('nan')
    ra = np.argsort(np.argsort(a)).astype(float)
    rb = np.argsort(np.argsort(b)).astype(float)
    return float(np.corrcoef(ra, rb)[0, 1])


pooled = [(p, r['select_mode']) for r in rows if r['keep_ratio'] < 1.0
          for p in r['per_image'] if p['min_line_cov'] is not None]
if len(pooled) >= 12:
    _rec = [p['recall'] for p, _ in pooled]
    print(f'\nQ3. Which statistic predicts per-image recall? (all pruned rows, n={len(pooled)})')
    print(f"  {'statistic':22s} {'family':12s} {'Spearman vs recall':>19s}")
    print('  ' + '-' * 56)
    for key, fam in (('retained_ink', 'aggregate'), ('mean_line_cov', 'aggregate'),
                     ('p10_line_cov', 'worst-case'), ('min_line_cov', 'worst-case')):
        print(f'  {key:22s} {fam:12s} '
              f'{spearman([p[key] for p, _ in pooled], _rec):19.3f}')
    rho_ink = spearman([p['retained_ink'] for p, _ in pooled], _rec)
    rho_min = spearman([p['min_line_cov'] for p, _ in pooled], _rec)
    if rho_min > rho_ink + 0.05:
        print('  => D2 CONFIRMED per-image: worst-case coverage predicts recall better '
              'than summed ink. Do NOT build a router objective on total retained '
              'saliency.')
    elif rho_ink > rho_min + 0.05:
        print('  => D2 FALSIFIED per-image: summed ink is the better predictor after '
              'all. D2 was a 4-row coincidence; revise it and re-check 1a/1d.')
    else:
        print('  => INCONCLUSIVE: the two families predict about equally well here, so '
              "D2's claim stands unconfirmed rather than refuted.")
    # Within-mode is the harder test: it removes the between-mode contrast that could
    # carry the pooled correlation on its own.
    print('\n  within-mode (controls for the mode -- image-to-image variation only):')
    for mode in ('router', 'negated', 'random', 'ink', 'stratified', 'stratified_negated'):
        sub_ = [p for p, m in pooled if m == mode]
        if len(sub_) >= 5:
            print(f"    {mode:19s} ink "
                  f"{spearman([p['retained_ink'] for p in sub_], [p['recall'] for p in sub_]):6.3f}"
                  f"   min-cov "
                  f"{spearman([p['min_line_cov'] for p in sub_], [p['recall'] for p in sub_]):6.3f}"
                  f"   (n={len(sub_)})")
else:
    print('\nQ3 skipped: too few per-image coverage records to correlate.')

with open(SELECTION_OUT, 'w') as f:
    json.dump({'meta': {'run6_reference': RUN6_REFERENCE,
                        'seed': SELECTION_SEED,
                        'max_len': SELECTION_MAX_LEN,
                        'control_drift_pts': drift,
                        'num_eval_samples': ctrl['num_eval_samples'],
                        'max_words': MAX_WORDS,
                        'harness_verified': harness_verified,
                        'harness_note': harness_note,
                        'harness_control': harness_row,
                        'control_ckpt': _ctrl_ckpt,
                        'provenance': PROVENANCE,
                        'complete': True}, 'rows': rows}, f, indent=2)
print(f'\nWrote {SELECTION_OUT}  ({(time.perf_counter() - _t0) / 60:.1f} min total)')


In [ ]:
# Cell 9: Package all results into a single downloadable zip in the Kaggle output folder
import os
import json
import zipfile

OUTPUT_DIR = '/kaggle/working'
ZIP_PATH = os.path.join(OUTPUT_DIR, 'pruned_ocr_results.zip')

# If the evaluation cell has run, persist its metrics alongside the artifacts
try:
    metrics = {
        'word_recall_pct': float(np.mean(recalls) * 100.0),
        'character_accuracy_pct': float((1.0 - np.mean(neds)) * 100.0),
        'word_order_pct': float(np.mean(word_orders) * 100.0),
        'mean_ned': float(np.mean(neds)),
        'avg_token_savings_pct': float(np.mean(compression_ratios)),
        'avg_latency_ms': float(np.mean(latencies)),
        'num_eval_samples': len(neds),
    }
    with open(os.path.join(OUTPUT_DIR, 'metrics.json'), 'w') as f:
        json.dump(metrics, f, indent=2)
    print('Wrote metrics.json:', metrics)
except NameError:
    print('Eval metrics not in scope (run the evaluation cell first) - zipping artifacts only.')

# Zip everything under /kaggle/working EXCEPT the archive itself and any prior .zip files
with zipfile.ZipFile(ZIP_PATH, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    file_count = 0
    for root, _, files in os.walk(OUTPUT_DIR):
        for fname in files:
            fpath = os.path.join(root, fname)
            if os.path.abspath(fpath) == os.path.abspath(ZIP_PATH):
                continue          # never zip the archive into itself
            if fname.lower().endswith('.zip'):
                continue          # skip any other zip archives (e.g. an old results zip)
            zf.write(fpath, os.path.relpath(fpath, OUTPUT_DIR))
            file_count += 1

size_mb = os.path.getsize(ZIP_PATH) / (1024 ** 2)
print(f'\nPackaged {file_count} files -> {ZIP_PATH} ({size_mb:.1f} MB)')
print('Download it from the Kaggle "Output" tab (or the /kaggle/working file browser).')
print('\nContents:')
with zipfile.ZipFile(ZIP_PATH) as zf:
    for info in zf.infolist():
        print(f'  {info.filename}  ({info.file_size / (1024**2):.2f} MB)')